# Embedding-Based Recommendation

## Objective

The baseline recommender represents each work using TF-IDF vectors derived from structured metadata. While effective, TF-IDF mainly captures exact keyword overlap and cannot model semantic relationships between features.

In this notebook, semantic embeddings are used to generate dense vector representations of each work. These embeddings capture the contextual meaning of the metadata, allowing semantically similar works to be positioned closer together in the embedding space.

The recommendation process follows the same principle as the baseline model:
1. Create a textual representation for each work.
2. Generate embeddings using a pre-trained sentence embedding model.
3. Capture cosine similarity between embedding vectors.
4. Recommend the most similar works based on semantic similarity.

The performance of embedding-based recommender will later be compared with TF-IDF baseline.

In [32]:
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

In [2]:
df = pd.read_csv(Path('../data/processed/classical_music_feature_engineered.csv'))

In [3]:
df.columns

Index(['title', 'subtitle', 'popular', 'recommended', 'work_id', 'genre',
       'composer_id', 'composer_name', 'birth', 'death', 'epoch',
       'title_subtitle', 'form', 'extended_genre', 'instrumentation', 'mode',
       'tempo_marking'],
      dtype='str')

### Create Text Descriptions

The embedding model requires textual input. Therefore, the extracted musical features are combined into natural language description.

Each work is represented by a short description containing available information such as epoch, form, genre, instruments, mode and tempo marking. Missing features are ignored to avoid adding incorrect information.

In [4]:
def create_description(row):
    parts = []

    if pd.notna(row['epoch']):
        parts.append(f"{row['epoch']}")

    if pd.notna(row['form']):
        parts.append(f"{row['form'].lower()}")
    else:
        parts.append('musical piece')

    description = 'A ' + ' '.join(parts)

    if pd.notna(row['instrumentation']):
        description += f" for {row['instrumentation'].lower()}"

    if pd.notna(row['mode']):
        description += f" in {row['mode'].lower()} mode"

    if pd.notna(row['extended_genre']):
        description += f", from the {row['extended_genre'].lower()} genre"

    if pd.notna(row['tempo_marking']):
        description += f", with {row['tempo_marking'].lower()} tempo"

    return description + "."

In [5]:
df['description'] = df.apply(create_description, axis=1)

In [6]:
df[['title','description']].sample(5)

,title,description
9457,"Sonata in B flat major, K.503, L.196","A Baroque sonata for piano in major mode, from..."
1727,"Lieder und Gesänge, op. 59","A Romantic musical piece for voice, piano, fro..."
7071,"Così dunque tradisci, K.432","A Classical musical piece for orchestra, from ..."
6443,"Oiseaux Exotiques, pour piano et orchestre","A 20th Century musical piece for piano, from t..."
8108,"Rapsodie nègre, for baritone and chamber ensem...","A 20th Century musical piece for voice, from t..."


### Generate Embeddings

The textual descriptions are converted into dense vector representations using a pre-trained sentence embedding model. Each embedding captures the semantic meaning of a musical work and will be used to compute similarity between works.

In [7]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [8]:
embedding = model.encode(df['description'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/407 [00:00<?, ?it/s]

In [9]:
embedding.shape

(13014, 384)

In [10]:
embedding_similarity_matrix = cosine_similarity(embedding)

In [11]:
embedding_similarity_matrix.shape

(13014, 13014)

### Save the Embedding Model

In [12]:
joblib.dump(embedding_similarity_matrix, Path('../models/embedding_similarity_matrix.pkl'))

['../models/embedding_similarity_matrix.pkl']